# VoiceGuard: fine-tune a deepfake-voice detector on a GPU

This notebook runs the **whole** training pipeline on a Colab GPU, for either model:

* **`xlsr_aasist`** (default, accuracy first): XLS-R 300M, pre-trained on 128 languages, with the top 8 of its 24 transformer layers fine-tuned plus an AASIST graph back-end.
* **`aasist_l`**: the 85K-parameter raw-waveform model that also runs on a phone.
 You do not upload any data:
every dataset is pulled straight from Hugging Face onto the Colab machine.

| Step | What happens | Where it lives |
|---|---|---|
| 1 | Clone the repo, install deps | `/content/voice-guard` |
| 2 | Download + preprocess all datasets (once) | cached as a tarball in Google Drive |
| 3 | Fine-tune the chosen model, checkpoint every epoch | Google Drive, so a disconnect loses nothing |
| 4 | Evaluate on unseen generators, export ONNX, plot | Google Drive |

**Before you start**

1. `Runtime → Change runtime type → GPU` (T4 works; L4 or A100 is faster).
2. Accept the terms of the gated datasets [ai4bharat/IndicVoices](https://huggingface.co/datasets/ai4bharat/IndicVoices) and [mueller91/MLAAD](https://huggingface.co/datasets/mueller91/MLAAD) with your Hugging Face account.
3. For `xlsr_aasist`, keep ~10 GB free in Google Drive (checkpoints are ~1.3 GB each, the ONNX export ~1.2 GB).
4. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a **read** token, and enable notebook access for it.
5. Only if you point `REPO_URL` at a **private** repo: add a second secret `GITHUB_TOKEN` (a GitHub token with read access to that repo).

Every step stops the notebook on failure, so the first red error is the real one.

If the session disconnects, reconnect and run all cells again: data is restored from Drive and training resumes from `last.pt`.

In [ ]:
#@title Settings
MODEL = "xlsr_aasist"  #@param ["xlsr_aasist", "aasist_l"]
REPO_URL = "https://github.com/Hashim-Bhagad/voiceguard.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
DRIVE_DIR = "/content/drive/MyDrive/voiceguard"  #@param {type:"string"}

# MLAAD supplies real Hindi deepfakes from 9 TTS systems: the only source here with
# Indic-language spoofs. It is CC-BY-NC-4.0: fine for coursework/research, not for a product.
ACCEPT_MLAAD_NONCOMMERCIAL = True  #@param {type:"boolean"}
ITW_MAX_PER_LABEL = 6000  #@param {type:"integer"}
INDIC_MAX_PER_LANG = 600  #@param {type:"integer"}
EPOCHS = 0  #@param {type:"integer"}
# 0 = use the config's default (12 for xlsr_aasist, 30 for aasist_l).
# Stop cleanly before Colab kills the session (free Colab: use ~3); rerun the notebook to resume.
MAX_HOURS = 10.5  #@param {type:"number"}

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
native_bf16 = torch.cuda.get_device_capability() >= (8, 0)  # T4/V100 only emulate bf16
print("torch", torch.__version__, "| cuda", torch.version.cuda,
      "| mixed precision:", "bf16" if native_bf16 else "fp16")

## 1. Code, dependencies and credentials

In [ ]:
import base64, os, pathlib, subprocess, sys
from google.colab import drive, userdata

os.environ["PYTHONUNBUFFERED"] = "1"  # child scripts print live


# Run a shell command, stream its output, and STOP the notebook if it fails.
# (`!cmd` in Colab ignores failures, so Run all would carry on with missing files.)
def sh(cmd, what=None):
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while chunk := proc.stdout.read1(4096):
        sys.stdout.write(chunk.decode(errors="replace"))
        sys.stdout.flush()
    if proc.wait():
        raise RuntimeError(f"{what or cmd} failed (exit code {proc.returncode}). The real error is in the output above.")


def secret(name, required=True):
    try:
        return userdata.get(name)
    except Exception as exc:  # missing secret, or notebook access switched off
        if required:
            raise RuntimeError(
                f"Colab secret {name!r} is missing or its 'Notebook access' toggle is off "
                "(key icon in the left sidebar)."
            ) from exc
        return None


drive.mount("/content/drive")
pathlib.Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

# A public repo needs no credentials. For a private one, add a GITHUB_TOKEN secret.
gh_token = secret("GITHUB_TOKEN", required=False)
auth = ""
if gh_token:
    basic = base64.b64encode(f"x-access-token:{gh_token}".encode()).decode()
    auth = f'-c http.https://github.com/.extraheader="AUTHORIZATION: basic {basic}"'

REPO = "/content/voice-guard"
if not os.path.isdir(f"{REPO}/.git"):
    sh(f"git {auth} clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}",
       what="git clone (private repo? add a GITHUB_TOKEN secret)")
else:
    sh(f"git {auth} -C {REPO} pull --ff-only", what="git pull")
TRAINING = f"{REPO}/training"
os.chdir(TRAINING)
print("code at", TRAINING, "@", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

# torch is already installed with CUDA on Colab; requirements-gpu.txt leaves it alone.
sh("pip install -q -r requirements-gpu.txt", what="pip install")

os.environ["HF_TOKEN"] = secret("HF_TOKEN")
os.environ["ACCEPT_MLAAD_NONCOMMERCIAL"] = "1" if ACCEPT_MLAAD_NONCOMMERCIAL else "0"
os.environ["ITW_MAX_PER_LABEL"] = str(ITW_MAX_PER_LABEL)
os.environ["INDIC_MAX_PER_LANG"] = str(INDIC_MAX_PER_LANG)

## 2. Data

The first run downloads and preprocesses every source (about 30-60 minutes, mostly the
~7 GB In-the-Wild archive), then saves the processed 16 kHz clips and the manifest to
Drive as one tarball. Later sessions restore that tarball in a few minutes instead.

In [ ]:
DATA_TAR = f"{DRIVE_DIR}/voiceguard-data.tar"
if os.path.isfile(DATA_TAR) and not os.path.isfile("data/metadata.csv"):
    print("restoring processed data from Drive ...")
    sh(f"tar -xf {DATA_TAR} -C {TRAINING}", what="restoring the data tarball")

sh("bash scripts/prepare_data.sh", what="data preparation")

if not os.path.isfile(DATA_TAR):
    print("saving processed data to Drive for next time ...")
    # Processed clips + manifest only: raw downloads (data/sources) and the decode cache are skipped.
    # Written under a temp name first so a half-written tarball is never mistaken for a good one.
    sh(f"tar -cf {DATA_TAR}.part -C {TRAINING} data/audio data/candidates data/metadata.csv data/metadata.build.json"
       f" && mv {DATA_TAR}.part {DATA_TAR}", what="saving the data tarball")
sh(f"du -sh {DATA_TAR}")

In [ ]:
import pandas as pd
df = pd.read_csv("data/metadata.csv")
display(df.groupby(["split", "label"]).size().unstack())
display(df.groupby(["language", "label"]).size().unstack(fill_value=0))
display(df[df.label == "spoof"].groupby(["split", "generator"]).size().rename("clips").to_frame())

## 3. Fine-tune

Checkpoints are written to Drive every epoch. If this cell is interrupted, rerun the
notebook from the top: it resumes from `last.pt` with optimiser and scheduler state intact.

In [ ]:
import yaml
CONFIG = {"xlsr_aasist": "configs/xlsr_aasist_gpu.yaml", "aasist_l": "configs/aasist_l_gpu.yaml"}[MODEL]
OUT_DIR = yaml.safe_load(open(CONFIG))["train"]["out_dir"]          # e.g. outputs/xlsr_aasist
ONNX_NAME = {"xlsr_aasist": "xlsr-aasist-indic-v0.1.onnx", "aasist_l": "aasist-l-indic-v0.2.onnx"}[MODEL]
RUN_DRIVE = f"{DRIVE_DIR}/{OUT_DIR}"
pathlib.Path(RUN_DRIVE).mkdir(parents=True, exist_ok=True)
RUN = f"{TRAINING}/{OUT_DIR}"
pathlib.Path(RUN).parent.mkdir(parents=True, exist_ok=True)
if not os.path.islink(RUN):
    sh(f"rm -rf {RUN}")
    os.symlink(RUN_DRIVE, RUN)

resume = os.path.isfile(f"{RUN}/last.pt")
if resume:
    start = "--resume"
elif MODEL == "aasist_l":
    start = "--pretrained weights/AASIST-L.pth"
else:
    start = ""  # XLS-R downloads facebook/wav2vec2-xls-r-300m itself
epochs = f"--epochs {EPOCHS}" if EPOCHS else ""
print("resuming from last.pt" if resume else f"starting a new {MODEL} run")
sh(f"python train_aasist_l.py --config {CONFIG} {start} {epochs} --max-hours {MAX_HOURS}", what="training")

## 4. Evaluate, export and report

In [ ]:
# Threshold from dev; headline EER on test, whose spoof generators were never trained on.
sh(f"python eval.py --ckpt {RUN}/best.pt --save-scores", what="evaluation")

In [ ]:
sh(f"python export_onnx.py --ckpt {RUN}/best.pt --config {CONFIG} --out {RUN}/{ONNX_NAME}", what="ONNX export")
sh(f"python scripts/inspect_finetune.py --config {CONFIG} --finetuned {RUN}/best.pt"
   f" --markdown {RUN}/finetune_report.md --json {RUN}/finetune_report.json", what="fine-tune report")
sh(f"python scripts/plot_results.py --run-dir {RUN}", what="plots")

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{RUN}/finetune_report.md").read()))
for name in ("training_curve", "roc_test", "score_distribution"):
    display(Image(f"{RUN}/figures/{name}.png"))

## 5. Use the new model

Everything is in `MyDrive/voiceguard/outputs/<model>/`. Both models export the same ONNX
contract (`waveform float32[batch, 64600]` in, `spoof_logit` out), so to serve either one, copy the
`.onnx` file into `backend/models/`, point `AASIST_L_ONNX` at it, set `MODEL_BACKEND=aasist-l-onnx`
in `backend/.env`, and restart the backend.

In [ ]:
sh(f"ls -lh {RUN} {RUN}/figures")